# Module 4: The Overfitting Trap
## Part 2: The Holdout Test — The Moment of Truth

---

In Part 1, you ran a parameter sweep and found a "winning" strategy.

It looked great. Positive returns. Good win rate. Multiple trades.

Now we're going to do something the sweep DIDN'T do:

**Test on data the sweep has never seen.**

This is called a **holdout test**. We reserve the last 20% of our data and never let the sweep touch it. Then we test the "winner" on that reserved data.

If the strategy is real, it should work on new data too.

If it was just lucky, it will fail.

---

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product

# Load the same sample dataset
df = pd.read_parquet(Path('data') / 'sample_eth_4h.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Total data: {len(df)} rows')
print(f'Date range: {df["timestamp"].min()} to {df["timestamp"].max()}')

### Step 1: Split the data — 80% for sweep, 20% holdout

The key rule: **the holdout data must NEVER be used during the sweep.**

In [ ]:
# Reserve last 20% as holdout
holdout_fraction = 0.20
split_idx = int(len(df) * (1 - holdout_fraction))

df_sweep = df.iloc[:split_idx]     # First 80% — for parameter selection
df_holdout = df.iloc[split_idx:]   # Last 20% — NEVER touched during sweep

print(f'Sweep data:   {len(df_sweep)} rows ({df_sweep["timestamp"].min().date()} to {df_sweep["timestamp"].max().date()})')
print(f'Holdout data: {len(df_holdout)} rows ({df_holdout["timestamp"].min().date()} to {df_holdout["timestamp"].max().date()})')
print()
print(f'The sweep will ONLY see data before {df_holdout["timestamp"].min().date()}')
print(f'The holdout starts AFTER that date — completely unseen data')

### Step 2: Re-run the sweep on ONLY the 80% data

In [ ]:
def build_splits(n_rows, train_bars=24*45, test_bars=24*14, step_bars=24*14):
    splits = []
    start = 0
    while start + train_bars + test_bars <= n_rows:
        splits.append({
            'train_start': start,
            'train_end': start + train_bars,
            'test_start': start + train_bars,
            'test_end': start + train_bars + test_bars,
        })
        start += step_bars
    return splits

def evaluate(data, splits, liq_z, liq_imb, crowd_cap, oi_div,
             target='fwd_ret_24', hold_bars=24, cost_bps=12):
    cost = cost_bps / 10000
    returns = []
    mask = (
        (data.get('liq_total_z_72', pd.Series(dtype=float)) > liq_z) &
        (data.get('liquidation_imbalance', pd.Series(dtype=float)) > liq_imb) &
        (data.get('crowding_top_pos_z_72', pd.Series(dtype=float)) < crowd_cap) &
        (data.get('oi_price_divergence_24', pd.Series(dtype=float)) > oi_div)
    ).fillna(False)
    
    for split in splits:
        test = data.iloc[split['test_start']:split['test_end']]
        tmask = mask.iloc[split['test_start']:split['test_end']]
        nxt = test.index.min()
        for idx in test.index:
            if idx < nxt or not bool(tmask.loc[idx]):
                continue
            if target in test.columns:
                val = test.at[idx, target]
                if not pd.isna(val):
                    returns.append(float(val) - cost)
            nxt = idx + hold_bars
    
    if not returns:
        return {'trades': 0, 'mean': 0, 'total': 0, 'win_rate': 0}
    r = np.array(returns)
    return {'trades': len(r), 'mean': float(r.mean()), 'total': float(r.sum()), 'win_rate': float((r > 0).mean())}

# Sweep on the 80% data only
sweep_splits = build_splits(len(df_sweep))
print(f'Walk-forward folds on sweep data: {len(sweep_splits)}')

param_grid = {
    'liq_z':    [0.3, 0.6, 1.0, 1.5, 2.0],
    'liq_imb':  [0.0, 0.05, 0.10, 0.20],
    'crowd':    [0.0, 0.2, 0.5, 0.7, 1.0],
    'oi_div':   [-0.10, -0.05, -0.02, 0.0],
}

combos = list(product(param_grid['liq_z'], param_grid['liq_imb'],
                       param_grid['crowd'], param_grid['oi_div']))

print(f'Testing {len(combos)} combinations on sweep data...')
results = []
for i, (lz, li, cc, od) in enumerate(combos):
    if (i+1) % 100 == 0: print(f'  {i+1}/{len(combos)}...')
    r = evaluate(df_sweep, sweep_splits, lz, li, cc, od)
    r.update({'liq_z': lz, 'liq_imb': li, 'crowd': cc, 'oi_div': od})
    results.append(r)

sweep_results = pd.DataFrame(results)
viable = sweep_results[sweep_results['trades'] >= 5].sort_values('total', ascending=False)

if len(viable) > 0:
    best = viable.iloc[0]
    print(f'\nSweep winner on 80% data:')
    print(f'  Total return: {best["total"]*100:+.2f}%')
    print(f'  Trades: {best["trades"]:.0f}')
    print(f'  Win rate: {best["win_rate"]*100:.0f}%')
    print(f'  Parameters: liq_z>{best["liq_z"]} imb>{best["liq_imb"]} crowd<{best["crowd"]} oi>{best["oi_div"]}')
else:
    print('No viable combinations found.')

### Step 3: THE HOLDOUT TEST

Now we take the winning parameters and test them on the 20% holdout data.

**This data was completely invisible during the sweep.**

If the strategy is real, it should work here too.

Ready?

In [ ]:
if len(viable) > 0:
    best = viable.iloc[0]
    
    # Test on holdout — simple: just run the strategy on the holdout period
    holdout_returns = []
    cost = 12 / 10000
    mask = (
        (df_holdout.get('liq_total_z_72', pd.Series(dtype=float)) > best['liq_z']) &
        (df_holdout.get('liquidation_imbalance', pd.Series(dtype=float)) > best['liq_imb']) &
        (df_holdout.get('crowding_top_pos_z_72', pd.Series(dtype=float)) < best['crowd']) &
        (df_holdout.get('oi_price_divergence_24', pd.Series(dtype=float)) > best['oi_div'])
    ).fillna(False)
    
    nxt = df_holdout.index.min()
    for idx in df_holdout.index:
        if idx < nxt or not bool(mask.loc[idx]):
            continue
        val = df_holdout.at[idx, 'fwd_ret_24']
        if not pd.isna(val):
            holdout_returns.append(float(val) - cost)
        nxt = idx + 24
    
    # THE REVEAL
    print('='*70)
    print('  THE MOMENT OF TRUTH: HOLDOUT TEST')
    print('='*70)
    print()
    print(f'  SWEEP RESULTS (data the parameters were chosen on):')
    print(f'    Total return:  {best["total"]*100:+.2f}%')
    print(f'    Trades:        {best["trades"]:.0f}')
    print(f'    Win rate:      {best["win_rate"]*100:.0f}%')
    print()
    
    if holdout_returns:
        hr = np.array(holdout_returns)
        print(f'  HOLDOUT RESULTS (data the parameters NEVER saw):')
        print(f'    Total return:  {hr.sum()*100:+.2f}%')
        print(f'    Trades:        {len(hr)}')
        print(f'    Win rate:      {(hr > 0).mean()*100:.0f}%')
    else:
        print(f'  HOLDOUT RESULTS: 0 trades generated')
        print(f'  The strategy found NO opportunities in unseen data.')
    
    print()
    print('='*70)
    
    if holdout_returns and np.array(holdout_returns).sum() > 0:
        print('  The strategy SURVIVED the holdout test.')
        print('  This doesn\'t guarantee it works forever,')
        print('  but it\'s a much stronger result than the sweep alone.')
    else:
        print('  The strategy FAILED the holdout test.')
        print()
        print('  The sweep "winner" does not work on new data.')
        print('  It was just the luckiest combination out of 400 tries.')
        print()
        print('  This is what happens to MOST sweep-optimized strategies.')
        print('  This is why most backtested trading systems fail in live trading.')

    print('='*70)

### Step 4: Visualize the gap

Let's see how ALL 400 combinations performed on the sweep vs the holdout.

In [ ]:
# Test ALL viable combinations on the holdout
if len(viable) > 0:
    comparison = []
    for _, row in viable.head(20).iterrows():  # Top 20 sweep winners
        mask = (
            (df_holdout.get('liq_total_z_72', pd.Series(dtype=float)) > row['liq_z']) &
            (df_holdout.get('liquidation_imbalance', pd.Series(dtype=float)) > row['liq_imb']) &
            (df_holdout.get('crowding_top_pos_z_72', pd.Series(dtype=float)) < row['crowd']) &
            (df_holdout.get('oi_price_divergence_24', pd.Series(dtype=float)) > row['oi_div'])
        ).fillna(False)
        
        h_rets = []
        nxt = df_holdout.index.min()
        cost = 12 / 10000
        for idx in df_holdout.index:
            if idx < nxt or not bool(mask.loc[idx]):
                continue
            val = df_holdout.at[idx, 'fwd_ret_24']
            if not pd.isna(val):
                h_rets.append(float(val) - cost)
            nxt = idx + 24
        
        h_total = sum(h_rets) if h_rets else 0
        comparison.append({
            'rank': len(comparison) + 1,
            'sweep_return': row['total'] * 100,
            'holdout_return': h_total * 100,
            'sweep_trades': row['trades'],
            'holdout_trades': len(h_rets),
        })
    
    comp_df = pd.DataFrame(comparison)
    
    print('Top 20 sweep winners vs their holdout performance:')
    print()
    print(f'{"Rank":<6} {"Sweep":>10} {"Holdout":>10} {"Gap":>10}  {"Verdict"}')
    print('-' * 55)
    for _, c in comp_df.iterrows():
        gap = c['holdout_return'] - c['sweep_return']
        verdict = 'SURVIVED' if c['holdout_return'] > 0 else 'FAILED'
        color = '' 
        print(f'{c["rank"]:<6.0f} {c["sweep_return"]:>+9.2f}% {c["holdout_return"]:>+9.2f}% {gap:>+9.2f}%  {verdict}')
    
    survived = (comp_df['holdout_return'] > 0).sum()
    total = len(comp_df)
    print(f'\nSurvival rate: {survived}/{total} ({survived/total*100:.0f}%)')
    print(f'\nThis is how many of your "best" strategies actually work on new data.')

---

## The Lesson

What you just saw is the **most common reason trading systems fail in real life.**

The process was:
1. Test 400 parameter combinations
2. Pick the best one
3. Report the results as if they represent the strategy's performance

But step 2 is where the lie happens. You're not testing a strategy — you're selecting the luckiest outcome from 400 independent trials.

### How to protect yourself

1. **Always use a holdout.** Reserve 20% of your data that no sweep, no optimization, no parameter selection ever sees.
2. **Report holdout results, not sweep results.** The only honest number is how the strategy performs on data it was never optimized on.
3. **Fewer parameters = less overfitting.** A strategy with 2 parameters is harder to overfit than one with 6.
4. **More trades = more reliable.** A strategy with 50 trades on the holdout is more trustworthy than one with 5.
5. **Cross-market validation.** If a strategy works on ETH AND BTC AND SOL, it's more likely real than one that only works on ETH.

### The golden rule

> **If you wouldn't bet your own money on the holdout results, the strategy doesn't work.**

---

In the next module, we'll look at what actually DOES survive holdout testing, and why structural edges (like funding carry) are fundamentally different from parameter-optimized signals.

→ Continue to **Module 5: What Doesn't Work (and Why)**